# JAX priors: numerical boundaries and original replay

Default prior calculations use JAX on the parameter device. This notebook measures differences, isolates their causes and proves original results with independent controls. Every fixture is generated here; no external data or test modules are required.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio", category=UserWarning)
warnings.filterwarnings("ignore", message="builtin type Swig.* has no __module__ attribute", category=DeprecationWarning)
import numpy as np
import scipy
from scipy import special
import jax
import jax.numpy as jnp
import jax.scipy.special as jsp
import pandas as pd
from IPython.display import display
from pycbc import distributions, transforms
from pycbc.scheme import JAXScheme
from pycbc.domain_jax import native_result

jax.config.update("jax_enable_x64", True)
device = "cuda" if any(d.platform == "gpu" for d in jax.devices()) else "cpu"
display(pd.DataFrame({"NumPy": [np.__version__], "SciPy": [scipy.__version__],
                      "JAX": [jax.__version__], "hardware": [jax.devices()[0].device_kind]}))

def exact(actual, expected):
    a, b = np.asarray(actual), np.asarray(expected)
    assert a.dtype == b.dtype and a.shape == b.shape
    assert a.tobytes() == b.tobytes()

def measure(label, actual, expected):
    a, b = np.asarray(actual), np.asarray(expected)
    return {"calculation": label, "JAX dtype": str(a.dtype), "original dtype": str(b.dtype),
            "different values": int(np.count_nonzero(a != b)),
            "max absolute difference": float(np.max(np.abs(a.astype(float) - b.astype(float))))}


,NumPy,SciPy,JAX,hardware
0,2.5.2,1.16.3,0.11.1,cpu


## Gaussian primitives and complete CDFs

Use the same values for SciPy and JAX error functions first. These are different primitive implementations. Then replay the unbounded CDF helper independently of the complete truncated CDF. Counts can be zero on a particular backend; exact agreement in a fixture is not a universal guarantee.

[JAX erf](https://docs.jax.dev/en/latest/_autosummary/jax.scipy.special.erf.html), [SciPy erf](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.erf.html).

In [2]:
z = np.random.default_rng(804).uniform(-2.5, 2.5, 64)
p = np.random.default_rng(805).uniform(0.01, 0.99, 64)
gaussian = distributions.Gaussian(x=(-2.0, 2.0), x_mean=0.25, x_var=1.5)
with JAXScheme(device):
    default_erf = jsp.erf(jnp.asarray(z))
    default_inverse = jsp.erfinv(jnp.asarray(2 * p - 1))
    default_cdf = gaussian.cdf("x", jnp.asarray(z))
with JAXScheme(device, reference_operations=("priors.Gaussian._normalcdf",)):
    original_helper = gaussian._normalcdf("x", jnp.asarray(z))
exact(original_helper, gaussian._normalcdf("x", z))
with JAXScheme(device, reference_operations=("priors.Gaussian.cdf",)):
    original_cdf = gaussian.cdf("x", jnp.asarray(z))
exact(original_cdf, gaussian.cdf("x", z))
display(pd.DataFrame([measure("erf primitive", default_erf, special.erf(z)),
                      measure("inverse-erf primitive", default_inverse, special.erfinv(2 * p - 1)),
                      measure("complete truncated CDF", default_cdf, original_cdf)]))


,calculation,JAX dtype,original dtype,different values,max absolute difference
0,erf primitive,float64,float64,23,2.220446e-16
1,inverse-erf primitive,float64,float64,43,4.440892e-16
2,complete truncated CDF,float64,float64,45,2.220446e-16


## One prior family at a time

A JAX broadcast grid is new API behavior. Its native log-PDF validation explicitly maps the original scalar implementation over that grid. Selecting the Gaussian leaves the power-law calculation active in JAX. Selecting both families validates their composition; the joint selector additionally replays original aggregation and constraints.

In [3]:
power = distributions.UniformPowerLaw(dim=3, r=(1.0, 4.0))
joint = distributions.JointDistribution(("x", "r"), gaussian, power)
x = np.array([[-0.5], [0.2], [0.9]])
r = np.array([[1.2, 2.2, 3.2]])
original_x = np.array([gaussian.logpdf(x=value) for value in x[:, 0]])[:, None]
with JAXScheme(device, reference_operations=("priors.Gaussian.logpdf",)):
    selected_gaussian = gaussian.logpdf(x=jnp.asarray(x))
    partial = joint(x=jnp.asarray(x), r=jnp.asarray(r))
    expected_partial = jnp.asarray(original_x) + power.logpdf(r=jnp.asarray(r))
exact(selected_gaussian, original_x)
exact(partial, expected_partial)
original_joint = np.array([[joint(x=xv, r=rv) for rv in r[0]] for xv in x[:, 0]])
with JAXScheme(device, reference_operations=("priors.Gaussian.logpdf", "priors.UniformPowerLaw.logpdf")):
    component_replay = joint(x=jnp.asarray(x), r=jnp.asarray(r))
with JAXScheme(device, reference_operations=("priors.JointDistribution.logpdf",)):
    complete_replay = joint(x=jnp.asarray(x), r=jnp.asarray(r))
exact(complete_replay, original_joint)
display(pd.DataFrame([measure("independent component replay", component_replay, original_joint),
                      measure("complete original joint replay", complete_replay, original_joint)]))


,calculation,JAX dtype,original dtype,different values,max absolute difference
0,independent component replay,float64,float64,0,0.0
1,complete original joint replay,float64,float64,0,0.0


## Cubic inverse CDF and angular functions

The mass-ratio inverse CDF shares the original 1000-point grid. SciPy evaluates its cubic spline representation; JAX uses a power basis and Horner evaluation. Angular inverses additionally call different inverse-trigonometric primitives. The original controls replay each complete inverse-CDF stage with its original dtype.

[Power-basis CubicSpline](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html).

In [4]:
mass = distributions.QfromUniformMass1Mass2(q=(1.0, 8.0))
angle = distributions.SinAngle(theta=(0.2, 0.9))
units = np.array([0.07, 0.23, 0.41, 0.69, 0.93], dtype=np.float32)
original_mass = mass.cdfinv(q=units)["q"]
original_angle = angle.cdfinv(theta=units)["theta"]
with JAXScheme(device):
    default_mass = mass.cdfinv(q=jnp.asarray(units))["q"]
    default_angle = angle.cdfinv(theta=jnp.asarray(units))["theta"]
with JAXScheme(device, reference_operations=("priors.QfromUniformMass1Mass2.cdfinv", "priors.SinAngle.cdfinv")):
    replay_mass = mass.cdfinv(q=jnp.asarray(units))["q"]
    replay_angle = angle.cdfinv(theta=jnp.asarray(units))["theta"]
exact(replay_mass, original_mass)
exact(replay_angle, original_angle)
display(pd.DataFrame([measure("mass-ratio inverse CDF", default_mass, original_mass),
                      measure("angular inverse CDF", default_angle, original_angle)]))


,calculation,JAX dtype,original dtype,different values,max absolute difference
0,mass-ratio inverse CDF,float32,float64,5,7.372373e-07
1,angular inverse CDF,float32,float64,5,1.219770e-07


## KDE: subtraction order, transforms and aggregation

The current JAX KDE subtracts samples before Cholesky whitening and uses log-sum-exp. The original SciPy density kernel whitens samples and queries separately, subtracts them and accumulates densities. This fixture isolates the whitening order with a large common offset.

The `priors.Arbitrary.kde` option runs that original fitted density kernel exactly, independently of logit, Jacobian and logarithm stages. [Original SciPy kernel](https://github.com/scipy/scipy/blob/v1.16.3/scipy/stats/_stats.pyx#L711-L794).

In [5]:
rng = np.random.default_rng(123)
samples = 1e8 + rng.normal(scale=0.01, size=64)
kde_prior = distributions.Arbitrary(x=samples)
queries = np.array([1e8 - 0.03, 1e8, 1e8 + 0.03])
original_density = kde_prior.kde.evaluate(queries)
with JAXScheme(device):
    default_kde_log = kde_prior.logpdf(x=jnp.asarray(queries))
with JAXScheme(device, reference_operations=("priors.Arbitrary.kde",)):
    native_density = native_result(
        "priors", "Arbitrary.kde", kde_prior.kde.evaluate, jnp.asarray(queries))
    selected_kde_log = kde_prior.logpdf(x=jnp.asarray(queries))
exact(native_density, original_density)
np.testing.assert_allclose(default_kde_log, np.log(original_density), rtol=1e-5, atol=4e-6)
display(pd.DataFrame({"query minus offset": queries - 1e8,
                      "original log density": np.log(original_density),
                      "default JAX log density": np.asarray(default_kde_log),
                      "original density with JAX logarithm": np.asarray(selected_kde_log)}))


,query minus offset,original log density,default JAX log density,original density with JAX logarithm
0,-0.03,-0.851648,-0.851651,-0.851648
1,0.00,3.676192,3.676192,3.676192
2,0.03,-1.035193,-1.035192,-1.035193


For a bounded KDE, replay logit and its Jacobian separately from the fitted density. The whole original `Arbitrary.logpdf` also keeps original dependency errors. NumPy 2.4+ rejects the original Python `float` conversion of a length-one density array; this notebook checks that limitation instead of changing the native API. The fitted-density boundary above remains valid.

[NumPy scalar conversion change](https://numpy.org/doc/stable/release/2.4.0-notes.html).

In [6]:
bounded_samples = np.random.default_rng(91).uniform(0.1, 0.9, size=64)
bounded_kde = distributions.Arbitrary(bounds={"y": (0.0, 1.0)}, y=bounded_samples)
values = np.array([0.2, 0.5, 0.8])
logit = transforms.Logit("y", "mapped", domain=(0.0, 1.0))
with JAXScheme(device, reference_operations=("transforms.Logit.logit", "transforms.Logit.jacobian")):
    transformed = logit.logit(jnp.asarray(values))
    jacobian = logit.jacobian({"y": jnp.asarray(values)})
exact(transformed, logit.logit(values))
original_jacobian = np.array([logit.jacobian({"y": value}) for value in values])
exact(jacobian, original_jacobian)
original_bounded_density = bounded_kde.kde.evaluate(logit.logit(values))
with JAXScheme(device, reference_operations=("priors.Arbitrary.kde", "transforms.Logit.logit", "transforms.Logit.jacobian")):
    composed = bounded_kde.logpdf(y=jnp.asarray(values))
    expected_composition = jnp.log(jnp.asarray(original_bounded_density)) + jnp.log(jnp.asarray(original_jacobian))
exact(composed, expected_composition)
try:
    original_public = np.array([bounded_kde.logpdf(y=value) for value in values])
except TypeError as original_error:
    with JAXScheme(device, reference_operations=("priors.Arbitrary.logpdf",)):
        try:
            bounded_kde.logpdf(y=jnp.asarray(values))
        except TypeError as replay_error:
            assert str(replay_error) == str(original_error)
        else:
            raise AssertionError("original dependency exception was lost")
    display(pd.DataFrame({"whole original public API": ["original scalar conversion limitation preserved"],
                          "independent density boundary": ["exact original bytes asserted"]}))
else:
    with JAXScheme(device, reference_operations=("priors.Arbitrary.logpdf",)):
        original_replay = bounded_kde.logpdf(y=jnp.asarray(values))
    exact(original_replay, original_public)
    display(pd.DataFrame({"whole original public API": ["exact original bytes asserted"]}))


,whole original public API,independent density boundary
0,original scalar conversion limitation preserved,exact original bytes asserted


## Tabulated priors and hull membership

The tabulated example shares tables prepared by the original integration/interpolation path. JAX performs device interpolation. A hull face is a discrete decision: float32 barycentric arithmetic can round differently from SciPy's double-precision search even with the same `100 * double eps` tolerance. The first stored point lies just outside the triangle in double precision; float32 barycentric summation can round its last coordinate to zero. Replay each stage independently, including all Boolean output bytes.

[Original Delaunay tolerance](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.Delaunay.find_simplex.html).

In [7]:
from pathlib import Path
from tempfile import TemporaryDirectory
import h5py

with TemporaryDirectory() as directory:
    density_path = Path(directory) / "density.txt"
    np.savetxt(density_path, np.column_stack(([0.0, 1.0, 2.0, 3.0], [1.0, 2.0, 2.0, 1.0])))
    table = distributions.DistributionFunctionFromFile(params=["x"], file_path=density_path, column_index=1)
    values = np.array([0.2, 0.9, 1.4, 2.7])
    original_logpdf = table.logpdf(x=values)
    probabilities = np.array([0.1, 0.3, 0.7, 0.9])
    original_inverse = table.cdfinv(x=probabilities)["x"]
    with JAXScheme(device, reference_operations=("priors.DistributionFunctionFromFile.logpdf", "priors.DistributionFunctionFromFile.cdfinv")):
        exact(table.logpdf(x=jnp.asarray(values)), original_logpdf)
        exact(table.cdfinv(x=jnp.asarray(probabilities))["x"], original_inverse)

    hull_path = Path(directory) / "coefficients.hdf5"
    with h5py.File(hull_path, "w") as handle:
        handle["coefficients"] = np.array([[0.0, 0.0], [0.1, 0.0], [0.0, 0.1]])
    hull = distributions.constraints.SupernovaeConvexHull("unused", principal_components_file=hull_path, hull_dimention=2)
    x_coefficients = np.array([0.002, 0.02], dtype=np.float32)
    y_coefficients = np.array([np.float32(0.1) - x_coefficients[0], 0.02], dtype=np.float32)
    coefficients = {"coeff_0": x_coefficients,
                    "coeff_1": y_coefficients,
                    "coeff_2": np.zeros(2, dtype=np.float32)}
    original_membership = hull(coefficients)
    with JAXScheme(device):
        default_membership = hull({key: jnp.asarray(value) for key, value in coefficients.items()})
    with JAXScheme(device, reference_operations=("priors.SupernovaeConvexHull.evaluate",)):
        original_membership_replay = hull({key: jnp.asarray(value) for key, value in coefficients.items()})
    exact(original_membership_replay, original_membership)
    display(pd.DataFrame({"stored x + y minus face (double)": x_coefficients.astype(float) + y_coefficients.astype(float) - 0.1,
                          "original hull membership": original_membership,
                          "default JAX membership": np.asarray(default_membership),
                          "original replay": np.asarray(original_membership_replay)}))


,stored x + y minus face (double),original hull membership,default JAX membership,original replay
0,4.982576e-09,False,True,False
1,-6.000000e-02,True,True,True


## Complete original composition and pairing

The category control composes original component priors, constraints and joint aggregation. Fixed samples independently retain the original selected row pairs. These assertions compare dtype, shape and bytes; they do not use tolerances.

In [8]:
constraint = distributions.constraints.Constraint("x < 1.0")
# Original construction estimates constraint normalization from host samples.
np.random.seed(41)
joint = distributions.JointDistribution(("x", "r"), gaussian, power,
                                        constraints=[constraint], n_test_samples=256)
xv = np.array([-0.5, 0.2, 1.4])
rv = np.array([1.2, 2.2, 3.2])
# Rejected rows intentionally take log(0), yielding minus infinity.
with np.errstate(divide="ignore"):
    expected_joint = np.array([joint(x=x, r=r) for x, r in zip(xv, rv)])
    with JAXScheme(device, reference_operations=("priors",)):
        full_original = joint(x=jnp.asarray(xv), r=jnp.asarray(rv))
        original_constraint = constraint({"x": jnp.asarray(xv)})
exact(full_original, expected_joint)
exact(original_constraint, constraint({"x": xv}))
fixed = distributions.FixedSamples(("x", "y"), {"x": np.array([1.0, 2.0, 3.0, 4.0]), "y": np.array([4.0, 3.0, 2.0, 1.0])})
u = np.array([0.1, 0.4, 0.7, 0.9])
expected_pairs = [fixed.cdfinv(x=a, y=b) for a, b in zip(u, u[::-1])]
with JAXScheme(device, reference_operations=("priors.FixedSamples.cdfinv",)):
    selected_pairs = fixed.cdfinv(x=jnp.asarray(u), y=jnp.asarray(u[::-1]))
for name in ("x", "y"):
    exact(selected_pairs[name], np.array([row[name] for row in expected_pairs]))
display(pd.DataFrame({"original joint log probability": expected_joint,
                      "replayed joint log probability": np.asarray(full_original)}))


,original joint log probability,replayed joint log probability
0,-3.670103,-3.670103
1,-2.271165,-2.271165
2,-inf,-inf


All original comparisons retain their dependency behavior. Default differences above are measurements for the displayed environment. Original controls intentionally use host copies and concrete inputs; disable them for JAX differentiation and device execution.